# Sensitivity analysis 1: patient bootstrap confidence intervals

Estimate **95% pointwise percentile confidence intervals** for AUC and balanced accuracy for all 127 NEWS2 subsets in four setting–outcome experiments, using **2,000 bootstrap samples from the original dataset**.

## Method

Each replicate samples N original patients with replacement before cohort/outcome filtering. The same sampled patients supply both settings and outcomes. Cross-validation and inner-fold balanced-accuracy threshold selection are repeated within every replicate. Copies of the same patient remain in a single fold. Metrics are mean outer-fold AUC and BA, matching the original aggregation; thresholds retain the original tie-breaking rule with reference 5.

Intervals are the 2.5th and 97.5th percentiles of replicate estimates. They are pointwise intervals for each prespecified subset, not simultaneous intervals or selection-adjusted intervals for a winning subset. Threshold-selection optimism is controlled through outer evaluation; the bootstrap additionally varies patient sampling and fold assignments.

**Execution:** prepared for local execution; no study analyses have been run while creating this notebook. Repeating 10-fold evaluation for 2,000 × 4 × 127 combinations is computationally intensive. Per-replicate checkpoints support resumption without changing seeds. The helper reuses the original evaluator rather than a new metric implementation.

## Eligibility

Missing FiO2 or any GCS component remains missing rather than being converted to a normal/abnormal category. Both notebooks use the same rule. This explicitly handles missingness and may differ from the original notebook’s `np.where` behavior. GCS totals outside 3–15 are treated as missing and excluded; inspect the eligibility audit before running. Unique patient IDs are required. If a replicate cannot support both outcome classes in every grouped fold, execution stops with its ID; replicates are never silently discarded.

**Resampling size:** 2,000 means 2,000 separate datasets, not 2,000 patients in one dataset. Each contains N draws with replacement from the N original patients. Eligibility filtering occurs after sampling. Every subset obtains 2,000 AUC estimates and 2,000 BA estimates per setting/outcome; each estimate is the mean of its outer-fold metrics. The 95% interval uses their 2.5th and 97.5th percentiles. Datasets are generated in memory, while their results are saved as separate replicate CSV files.

## 1. Setup and configuration

In [5]:
from pathlib import Path
import sys
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Supports starting Jupyter from the repository root or this notebook folder.
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "functions" / "news2_scale_deconstruction.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Start Jupyter from the NEWS2 repository or sensitivity folder.")
SENSITIVITY_DIR = ROOT / "sensitivity analisis"
for location in (ROOT, SENSITIVITY_DIR):
    if str(location) not in sys.path:
        sys.path.insert(0, str(location))

from sensitivity_helpers import (
    FEATURES, OUTCOMES, load_original_data, prepare_cohort,
    assign_grouped_folds, run_patient_bootstrap, audit_gcs,
)

DATA_PATH = ROOT / "data" / "NEWs2.csv"
RANDOM_STATE = 42
N_OUTER_FOLDS = 10
N_JOBS = os.cpu_count() or 1  # Concurrent whole-dataset worker processes.
print(f"Using {N_JOBS} concurrent worker processes.")

N_BOOTSTRAPS = 2000
SUMMARY_DIR = ROOT / "data" / "outputs" / "sensitivity" / "boostrap"
OUTPUT_DIR = SUMMARY_DIR / "bootstrap_2000_valid_gcs"


Using 10 concurrent worker processes.


## 2. Load the original patient dataset

Resampling uses this original dataset, not a saved result table or previously bootstrapped dataset.

In [6]:
raw_data = load_original_data(DATA_PATH)
pd.DataFrame({"original_patients": [len(raw_data)], "bootstrap_replicates": [N_BOOTSTRAPS]})

,original_patients,bootstrap_replicates
0,18001,2000


## 3. Run or resume the bootstrap

The initial estimate and all replicates use the same grouped CV procedure. Progress counts completed replicates; each contains all four experiments and 127 subsets. Keep configuration and dataset unchanged when resuming. Use a new output directory for changed settings.

**Parallelism:** `N_JOBS` is the maximum number of complete bootstrap datasets evaluated simultaneously, using separate processes. Each worker runs all four experiments and their 127 subsets sequentially, without nested subset parallelism. Progress counts fully completed, saved datasets and displays the active worker count. Each worker holds its own original dataset and resample in memory. The original-data estimate is computed once before bootstrap workers start.

Restart the kernel after updating the helper. Previously completed replicate files remain reusable with the same data/configuration. Seeds are tied to dataset IDs, so results do not depend on completion order. Interrupting waits for already running tasks to stop before returning; restarting the kernel terminates its workers. Only complete saved datasets are reused.

In [7]:
bootstrap_summary = run_patient_bootstrap(
    raw_data, OUTPUT_DIR,
    n_bootstraps=N_BOOTSTRAPS,
    n_splits=N_OUTER_FOLDS,
    random_state=RANDOM_STATE,
    n_jobs=N_JOBS,
)
bootstrap_summary.head()

Completed bootstrap datasets: 100%|##########| 2000/2000 [00:00<?, ?dataset/s]

Reading checkpoints:   0%|          | 0/2000 [00:00<?, ?file/s]

,subset_key,subset_label,features_json,n_features,auc_estimate,ba_estimate,threshold_mean,cohort,outcome,n_patients_with_copies,n_unique_patients,bootstrap_id,auc_ci95_low,auc_ci95_high,ba_ci95_low,ba_ci95_high,n_bootstraps
0,resp_rate,resp_rate,"[""resp_rate""]",1,0.663633,0.644516,1.0,Prehospital,M30D,17970,17970,-1,0.652567,0.675733,0.629659,0.655509,2000
1,blod_oxy,blod_oxy,"[""blod_oxy""]",1,0.712459,0.683255,2.0,Prehospital,M30D,17970,17970,-1,0.700531,0.724135,0.670980,0.694206,2000
2,oxigen,oxigen,"[""oxigen""]",1,0.604826,0.604826,2.0,Prehospital,M30D,17970,17970,-1,0.595356,0.615140,0.595356,0.615140,2000
3,blod_pres,blod_pres,"[""blod_pres""]",1,0.607220,0.596728,2.0,Prehospital,M30D,17970,17970,-1,0.595585,0.619002,0.582684,0.607557,2000
4,heart_rate,heart_rate,"[""heart_rate""]",1,0.587372,0.570628,1.0,Prehospital,M30D,17970,17970,-1,0.574915,0.599560,0.553640,0.581338,2000


## 4. Full-score intervals

The complete CSV includes all 508 experiment–subset combinations. Displayed intervals are 95% percentile confidence intervals.

In [8]:
columns = ["cohort", "outcome", "n_features", "auc_estimate",
           "auc_ci95_low", "auc_ci95_high", "ba_estimate", "ba_ci95_low", "ba_ci95_high"]
bootstrap_summary.loc[bootstrap_summary["n_features"].eq(7), columns]

,cohort,outcome,n_features,auc_estimate,auc_ci95_low,auc_ci95_high,ba_estimate,ba_ci95_low,ba_ci95_high
126,Prehospital,M30D,7,0.814641,0.805245,0.824259,0.736329,0.724043,0.745595
253,Prehospital,UVI,7,0.758027,0.748345,0.766879,0.695531,0.682386,0.703621
380,Hospital,M30D,7,0.861860,0.853103,0.870570,0.784196,0.771158,0.793567
507,Hospital,UVI,7,0.800884,0.792370,0.809979,0.751868,0.743603,0.760853


## Outputs and interpretation

`bootstrap_95ci_all_subsets.csv` contains original grouped-CV estimates and 95% intervals. `bootstrap_0000.csv` through `bootstrap_1999.csv` retain replicate metrics. `bootstrap_settings.json` records data fingerprint and configuration. Intervals quantify uncertainty under this resampling and evaluation procedure; they do not establish significance of between-subset differences.

Run all cells locally after supplying the original dataset. No confidence limits or conclusions are available until execution completes.

### Output layout

Paths are relative to the NEWS2 repository root:

```text
data/outputs/sensitivity/boostrap/
├── original_estimates.csv
├── bootstrap_95ci_all_subsets.csv
└── bootstrap_2000_valid_gcs/
    ├── bootstrap_settings.json
    ├── bootstrap_0000.csv
    ├── ...
    └── bootstrap_1999.csv
```

`OUTPUT_DIR` is the lower directory containing replicate checkpoints. `SUMMARY_DIR` is its parent. Existing files are not moved by this code edit; to resume previous results, relocate the existing replicate CSVs and their settings manifest into the lower directory, and `original_estimates.csv` into its parent, before resuming.

## 6. Additional bootstrap statistics for the manuscript and report

This section reads the **existing** 2,000 replicate CSVs and `original_estimates.csv`; it never generates or evaluates another dataset. Every new CSV is saved in `SUMMARY_DIR` (`data/outputs/sensitivity/boostrap/`), one level above the replicate files.

### Estimand and interpretation

The manuscript describes whole-cohort AUC. Existing bootstrap files instead contain **mean outer-fold AUC** and nested-CV balanced accuracy. All results below retain those definitions. Do not attach these intervals to a whole-cohort AUC estimate. Check `bootstrap_report_calculation_status.csv` for manuscript items that cannot be reconstructed from saved results.

- **Paired subset-minus-full differences:** 95% percentile intervals for ΔAUC and ΔBA, pairing within the same bootstrap dataset. A subset meets the stated descriptive noninferiority criterion when its ΔAUC lower bound exceeds −0.02. These intervals are pointwise; testing all subsets is not multiplicity-adjusted.
- **Shapley attribution:** 95% intervals in AUC units and percentage of full-score AUC above chance. Negative contributions are retained. Replicates where full AUC ≤0.5 cannot yield meaningful normalized percentages; their counts are reported explicitly. If present, percentage intervals are conditional on positive denominators.
- **Parameter removal:** intervals for every one of the 63 matched removal pairs per parameter, plus intervals for their mean and median, overall and by parent subset size. Parent size 7 gives removal from the full score. An interval for the mean/median effect differs from the distribution of the 63 context effects in the boxplot.
- **Dominance stability:** fraction of bootstrap datasets where strict complete dominance holds in all 32 common contexts. Also report percentile intervals for the within-replicate fraction of contexts won. Stability is not a posterior probability or significance test.
- **Selection frequencies:** repeat best-AUC and best-BA selection within every bootstrap, overall and for each subset size. Ties use the other metric, then fewer parameters, then lexical subset key. Intervals for the subset chosen on the original data remain **fixed-subset intervals**, not selection-adjusted intervals.

### Items requiring a different calculation

DeLong tests and the manuscript's Harrell selection-optimism correction cannot be recovered from these saved CV summary tables. Neither can sensitivity/specificity intervals (not retained in existing exports) or raw-change AUC intervals. They are recorded as unavailable rather than approximated using a different statistic. This section does not change or rerun existing bootstraps.

The manuscript also describes GCS ≤13 in its sensitivity analysis, whereas the requested notebook implements GCS <13; resolve this wording before publication.

In [9]:
from bootstrap_report_intervals import load_report_bootstrap, export_report_bootstrap

# SUMMARY_DIR is OUTPUT_DIR.parent. Existing replicate files are read only.
report_bootstrap_data = load_report_bootstrap(
    OUTPUT_DIR,
    n_bootstraps=N_BOOTSTRAPS,
)

Load saved bootstrap datasets:   0%|          | 0/2000 [00:00<?, ?CSV/s]

In [10]:
report_csv_files = export_report_bootstrap(
    report_bootstrap_data,
    noninferiority_margin=0.02,
)
report_csv_files

Summarize report statistics:   0%|          | 0/4 [00:00<?, ?experiment/s]

,output,csv_path
0,paired_differences_95ci,/Users/aimarbarrenapol/Documents/NEWS2/data/ou...
1,shapley_auc_95ci,/Users/aimarbarrenapol/Documents/NEWS2/data/ou...
2,parameter_removal_pairs_95ci,/Users/aimarbarrenapol/Documents/NEWS2/data/ou...
3,parameter_removal_summary_95ci,/Users/aimarbarrenapol/Documents/NEWS2/data/ou...
4,dominance_bootstrap_stability,/Users/aimarbarrenapol/Documents/NEWS2/data/ou...
5,subset_selection_frequencies,/Users/aimarbarrenapol/Documents/NEWS2/data/ou...
6,selected_subsets_95ci,/Users/aimarbarrenapol/Documents/NEWS2/data/ou...
7,calculation_status,/Users/aimarbarrenapol/Documents/NEWS2/data/ou...


### CSV outputs

- `paired_differences_95ci.csv`: all subset–full ΔAUC/ΔBA intervals and pointwise noninferiority flags.
- `shapley_auc_95ci.csv`: absolute and percentage attribution intervals.
- `parameter_removal_pairs_95ci.csv`: individual removal-pair effects and intervals.
- `parameter_removal_summary_95ci.csv`: mean/median removal effects and intervals by parameter and parent size.
- `dominance_bootstrap_stability.csv`: complete-dominance stability and context-win-fraction intervals.
- `subset_selection_frequencies.csv`: selection counts/frequencies overall and within each subset size.
- `selected_subsets_95ci.csv`: full score, singletons, core, and originally selected subsets with fixed-subset AUC/BA intervals.
- `bootstrap_report_calculation_status.csv`: available calculations and manuscript mismatches.

Run these new cells locally after bootstrap completion. No summary statistics or tests were executed while authoring this section.